# Hypothetical Prompt Embeddings (HyPE) — Deep Dive

## Overview

**Hypothetical Prompt Embeddings (HyPE)** is a retrieval-augmentation technique that
*inverts* the HyDE paradigm: instead of generating a hypothetical **document** at query
time, HyPE generates hypothetical **questions** at *indexing* time for every chunk in the
corpus.  At query time the user's question is matched against those pre-computed question
embeddings, turning retrieval into a **question → question** similarity problem.

This notebook builds the entire HyPE pipeline from scratch — no LangChain, no LlamaIndex,
no OpenAI API — using **Qwen3-8B** (4-bit) for generation and **BGE-base-en-v1.5** for
embeddings, with **FAISS** as the vector store.

### Roadmap

| # | Section | Key idea |
|---|---------|----------|
| 1 | HyDE Recap | Quick refresher on the "forward" approach |
| 2 | What is HyPE? | The inverse — pre-compute questions, not answers |
| 3 | Why HyPE Works | Semantic-space alignment argument |
| 4 | The HyPE Algorithm | Step-by-step walkthrough |
| 5 | Implementation from Scratch | Full pipeline |
| 6 | Question Quality Analysis | Diversity & coverage |
| 7 | Comparison: Standard vs HyDE vs HyPE | Side-by-side retrieval evaluation |
| 8 | Hybrid Approach | Fusing document + question indices |
| 9 | Synthesis & Tradeoffs | When to use what |

## Setup

Install dependencies, mount Google Drive for model caching, and load:

* **Qwen/Qwen3-8B** (NF4 4-bit) — local LLM for question & document generation
* **BAAI/bge-base-en-v1.5** — embedding model via `sentence-transformers`
* **FAISS** — CPU-based vector index

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch \
    sentence-transformers faiss-cpu numpy

import torch, numpy as np, faiss, time, textwrap, json, re
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/models"

# ── LLM ──
MODEL_NAME = "Qwen/Qwen3-8B"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

# ── Embedding model ──
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)

print(f"LLM loaded: {MODEL_NAME}  (device: {llm.device})")
print(f"Embedding dim: {embed_model.get_sentence_embedding_dimension()}")

In [ ]:
def generate(messages, max_new_tokens=512, temperature=0.7):
    """Send a chat-message list to the LLM and return the assistant reply."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(llm.device)
    out = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_k=20,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Quick smoke test
print(generate([{"role": "user", "content": "Say 'HyPE is ready!' in exactly three words."}],
               max_new_tokens=20, temperature=0.3))

## Synthetic Knowledge Base

We build a small but realistic corpus of **six documents** covering machine learning
topics.  Each document is ~200-400 words — large enough for meaningful chunk-level
question generation, small enough to fit in a notebook demo.

In [ ]:
KNOWLEDGE_BASE = [
    {
        "title": "Gradient Descent Optimization",
        "text": (
            "Gradient descent is the workhorse optimization algorithm behind most modern machine "
            "learning. The algorithm iteratively updates model parameters by moving in the direction "
            "of steepest descent of the loss function. The learning rate controls the step size: too "
            "large and the optimizer overshoots minima; too small and convergence is painfully slow. "
            "Stochastic Gradient Descent (SGD) approximates the true gradient using a mini-batch of "
            "examples, which introduces noise but dramatically speeds up each iteration. Momentum "
            "methods like SGD-with-momentum accumulate a velocity vector that smooths out oscillations "
            "across ravines. Adam (Adaptive Moment Estimation) combines momentum with per-parameter "
            "adaptive learning rates using exponential moving averages of the first and second moments "
            "of the gradient. AdaGrad adapts learning rates based on the historical sum of squared "
            "gradients, working well for sparse features but decaying too aggressively for dense ones. "
            "RMSProp fixes AdaGrad's decay by using a moving average instead of a cumulative sum. "
            "Learning-rate scheduling — step decay, cosine annealing, warm restarts — is critical for "
            "reaching good minima in practice. Gradient clipping prevents exploding gradients in deep "
            "networks by capping the gradient norm."
        ),
    },
    {
        "title": "Transformer Architecture",
        "text": (
            "The Transformer architecture, introduced in the 2017 paper 'Attention Is All You Need', "
            "replaced recurrence with self-attention, enabling massive parallelism during training. "
            "Each layer consists of multi-head self-attention followed by a position-wise feed-forward "
            "network, with residual connections and layer normalization around each sub-layer. "
            "Self-attention computes Query, Key, and Value matrices from the input; attention scores "
            "are obtained via scaled dot-product of Q and K, followed by softmax and multiplication "
            "with V. Multi-head attention runs several attention heads in parallel, allowing the model "
            "to jointly attend to information from different representation subspaces. Positional "
            "encodings (sinusoidal or learned) inject sequence-order information since the architecture "
            "has no built-in notion of position. The encoder stack processes the full input sequence "
            "bidirectionally, while the decoder stack uses masked self-attention to preserve auto-"
            "regressive generation. Cross-attention layers in the decoder attend over encoder outputs. "
            "Pre-norm variants move layer normalization before the attention and feed-forward blocks, "
            "which stabilizes training of very deep models."
        ),
    },
    {
        "title": "Retrieval-Augmented Generation",
        "text": (
            "Retrieval-Augmented Generation (RAG) enhances large language models by grounding them in "
            "external knowledge retrieved at inference time. The canonical RAG pipeline has three stages: "
            "indexing, retrieval, and generation. During indexing, documents are chunked and embedded "
            "into a vector store (e.g., FAISS). At query time the user question is embedded and the "
            "top-k nearest chunks are retrieved via approximate nearest neighbor search. These chunks "
            "are concatenated into a context window and passed alongside the question to the LLM for "
            "answer generation. Key challenges include the semantic gap between short queries and long "
            "documents, chunk-boundary artifacts, and hallucination when the retrieved context is "
            "insufficient. Advanced techniques include re-ranking retrieved passages, hybrid "
            "sparse-dense retrieval (BM25 + vector), query rewriting, HyDE (hypothetical document "
            "embeddings), and multi-hop retrieval for complex questions. Evaluation metrics include "
            "retrieval precision, recall, MRR (Mean Reciprocal Rank), and downstream answer quality "
            "assessed via faithfulness, relevance, and completeness rubrics."
        ),
    },
    {
        "title": "Convolutional Neural Networks",
        "text": (
            "Convolutional Neural Networks (CNNs) are specialized deep learning architectures designed "
            "for grid-structured data such as images. A CNN applies learnable filters (kernels) that "
            "slide across the input, computing local dot products to produce feature maps. Pooling "
            "layers — typically max-pooling or average-pooling — downsample these maps, reducing "
            "spatial dimensions while retaining the most salient features. Stacking convolution and "
            "pooling layers creates a hierarchy: early layers detect edges and textures, middle layers "
            "capture parts and motifs, and deep layers encode high-level object semantics. Batch "
            "normalization after each convolution stabilizes training by normalizing activations. "
            "Residual connections (ResNet) allow gradients to flow through skip connections, enabling "
            "training of networks with hundreds of layers. Depthwise separable convolutions "
            "(MobileNet) factorize standard convolutions into a depthwise and a pointwise step, "
            "drastically reducing parameters and computation. Transfer learning with pre-trained CNNs "
            "(ImageNet weights) is standard practice: freeze early layers and fine-tune deeper layers "
            "for the target task. Data augmentation — random crops, flips, color jitter — is essential "
            "to prevent overfitting on small datasets."
        ),
    },
    {
        "title": "Regularization Techniques",
        "text": (
            "Regularization prevents machine learning models from overfitting to training data. "
            "L2 regularization (weight decay) adds a penalty proportional to the squared magnitude of "
            "weights, encouraging smaller, more distributed parameters. L1 regularization promotes "
            "sparsity by penalizing the absolute values, effectively performing feature selection. "
            "Dropout randomly zeroes activations during training with probability p, forcing the "
            "network to learn redundant representations. At test time, activations are scaled by (1-p) "
            "or, equivalently, inverted dropout scales during training. Batch normalization has a "
            "regularizing effect because the mini-batch statistics introduce noise. Early stopping "
            "monitors validation loss and halts training when it stops improving, preventing the model "
            "from memorizing noise in the training data. Data augmentation is an implicit regularizer "
            "that increases the effective training-set size. Label smoothing replaces hard 0/1 targets "
            "with soft targets (e.g., 0.9/0.1), reducing overconfident predictions and improving "
            "calibration. Mixup and CutMix create synthetic training examples by blending images and "
            "labels. Weight tying in language models forces the input and output embeddings to share "
            "parameters, reducing the total parameter count while improving generalization."
        ),
    },
    {
        "title": "Evaluation Metrics for Classification",
        "text": (
            "Choosing the right evaluation metric is critical for understanding model performance. "
            "Accuracy — the fraction of correct predictions — is misleading on imbalanced datasets. "
            "Precision measures the fraction of predicted positives that are truly positive, while "
            "recall (sensitivity) measures the fraction of actual positives correctly identified. The "
            "F1 score is the harmonic mean of precision and recall, balancing both concerns. The "
            "confusion matrix provides a complete picture: true positives, false positives, true "
            "negatives, and false negatives. The ROC curve plots the true positive rate against the "
            "false positive rate at varying classification thresholds; the Area Under the ROC Curve "
            "(AUC-ROC) summarizes discriminative ability. Precision-Recall curves are preferred when "
            "the positive class is rare. Cohen's Kappa measures inter-rater agreement adjusted for "
            "chance. For multi-class problems, metrics can be macro-averaged (each class weighted "
            "equally), micro-averaged (each sample weighted equally), or weighted by class prevalence. "
            "Log loss (cross-entropy) penalizes confident wrong predictions heavily, making it a "
            "preferred training objective and evaluation metric for probabilistic classifiers."
        ),
    },
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
for doc in KNOWLEDGE_BASE:
    print(f"  • {doc['title']}  ({len(doc['text'].split())} words)")

## Chunking

We split each document into overlapping chunks of ~200 characters with a 50-character
overlap.  This is intentionally small so that we get multiple chunks per document,
exercising the multi-question-per-chunk logic later.

In [ ]:
def chunk_text(text, chunk_size=200, overlap=50):
    """Split text into overlapping character-level chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []
chunk_metadata = []  # parallel list: which doc each chunk came from
for doc in KNOWLEDGE_BASE:
    doc_chunks = chunk_text(doc["text"])
    for c in doc_chunks:
        all_chunks.append(c)
        chunk_metadata.append(doc["title"])

print(f"Total chunks: {len(all_chunks)}")
for i, (chunk, title) in enumerate(zip(all_chunks[:5], chunk_metadata[:5])):
    print(f"\nChunk {i} [{title}]:")
    print(textwrap.fill(chunk, width=90))

---
# Part 1 — HyDE Recap

**Hypothetical Document Embedding (HyDE)** is a query-time technique:

1. The user asks a question.
2. An LLM generates a *hypothetical answer* (a fake but plausible document).
3. That hypothetical document is embedded.
4. The embedding is used to search the vector store.

The intuition is that a document-length text is closer in embedding space to real documents
than a short question would be.

### HyDE's limitations

| Limitation | Impact |
|------------|--------|
| **Query-time cost** | Every query requires an LLM call *before* retrieval begins |
| **Latency** | Adds 1-5 seconds per query depending on model size |
| **Hallucination risk** | If the LLM fabricates facts, the hypothetical document may steer retrieval toward wrong passages |
| **Not parallelizable** | Each user query is a serial LLM → embed → search chain |

Let's implement a minimal HyDE for comparison purposes.

In [ ]:
# ── Embed all chunks for Standard RAG and HyDE ──
print("Embedding all chunks with BGE …")
chunk_embeddings = embed_model.encode(all_chunks, show_progress_bar=True, normalize_embeddings=True)
dim = chunk_embeddings.shape[1]

# Build FAISS index for standard retrieval
standard_index = faiss.IndexFlatIP(dim)  # inner product (cosine after normalization)
standard_index.add(chunk_embeddings.astype(np.float32))

print(f"Standard FAISS index: {standard_index.ntotal} vectors, dim={dim}")

def standard_retrieve(query, k=3):
    """Standard RAG: embed query, search chunk index."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = standard_index.search(q_emb, k)
    return [(all_chunks[i], chunk_metadata[i], float(scores[0][j]))
            for j, i in enumerate(indices[0])]

# ── HyDE retrieve ──
def hyde_retrieve(query, k=3):
    """HyDE: generate hypothetical doc, embed it, search chunk index."""
    prompt = (
        f"Write a short, detailed paragraph that directly answers this question. "
        f"Do not repeat the question.\n\nQuestion: {query}"
    )
    messages = [{"role": "user", "content": prompt}]
    hypo_doc = generate(messages, max_new_tokens=200, temperature=0.5)
    hypo_emb = embed_model.encode([hypo_doc], normalize_embeddings=True).astype(np.float32)
    scores, indices = standard_index.search(hypo_emb, k)
    results = [(all_chunks[i], chunk_metadata[i], float(scores[0][j]))
               for j, i in enumerate(indices[0])]
    return results, hypo_doc

# Quick test
results = standard_retrieve("What is the learning rate in gradient descent?")
print("\n--- Standard Retrieval (top-1) ---")
print(f"  [{results[0][1]}] score={results[0][2]:.4f}")
print(f"  {results[0][0][:120]}…")

---
# Part 2 — What is HyPE?

## The Inverse of HyDE

HyPE (**Hypothetical Prompt Embeddings**) flips the HyDE idea on its head:

| | HyDE | HyPE |
|---|------|------|
| **When** | Query time | Indexing time |
| **What LLM generates** | A hypothetical *answer* document | Hypothetical *questions* |
| **What gets embedded** | The hypothetical document | The hypothetical questions |
| **Search compares** | Fake-doc embedding → real-doc embeddings | Real query embedding → fake-question embeddings |
| **LLM calls at query time** | 1 per query | **0** |
| **LLM calls at index time** | 0 | 1 per chunk |

### Core Insight

A user's query is a **question**.  If we pre-generate questions that each chunk could
answer, and embed those questions, then at query time we are comparing **question ↔ question**
— texts that live in the same semantic register.  This eliminates the style mismatch between
a terse user question and a verbose document passage.

### Analogy

Think of it like a library card catalog.  Instead of searching the full text of every book
(standard RAG) or writing a fake book summary and looking for similar summaries (HyDE), we
write a set of questions on each catalog card: *"What topics does this book cover?"*  When
someone comes in with a question, we scan the catalog cards — matching **question to question**
is more natural than matching a question to raw book text.

---
# Part 3 — Why HyPE Works

## Semantic Space Alignment

Embedding models map text into a high-dimensional space where semantically similar texts
cluster together.  But "similar" depends on *register*:

* A question like *"How does dropout prevent overfitting?"* lives in **interrogative space** —
  short, direct, seeking.
* A passage like *"Dropout randomly zeroes activations during training…"* lives in
  **declarative/expository space** — longer, descriptive, explaining.

Even though they're *about* the same topic, the surface forms differ enough that cosine
similarity can fail to rank the passage highly.

### HyPE's fix

By generating questions like *"What does dropout do during training?"* for the dropout chunk,
we place a **proxy** in interrogative space right next to where real user queries will land.
The retrieval comparison becomes:

```
cos(embed("How does dropout prevent overfitting?"),
    embed("What does dropout do during training?"))
```

Both are questions — same register, same length distribution, same syntactic structure.
Cosine similarity works much better in this regime.

### Empirical evidence

The HyPE paper (SSRN preprint 5139335) reports:
- Up to **42 percentage-point** improvement in retrieval precision
- Up to **45 percentage-point** improvement in claim recall

These gains come precisely from this semantic-space alignment.

---
# Part 4 — The HyPE Algorithm

## Step-by-Step

```
INDEXING PHASE (offline, one-time)
──────────────────────────────────
1.  Chunk the corpus into passages  C₁, C₂, …, Cₘ

2.  For each chunk Cᵢ:
      a.  Prompt the LLM:
          "Generate N diverse questions that this passage could answer."
      b.  Parse the N questions  q_{i,1}, q_{i,2}, …, q_{i,N}
      c.  Embed each question → v_{i,j} = embed(q_{i,j})
      d.  Store (v_{i,j}, pointer_to_Cᵢ) in the FAISS index

3.  The index now contains  M × N  question vectors,
    each pointing back to its source chunk.

QUERY PHASE (online, per-query)
───────────────────────────────
4.  User submits query Q

5.  Embed the query → v_Q = embed(Q)

6.  Search the FAISS index for the top-k nearest question vectors

7.  Collect the unique source chunks for those questions

8.  Return chunks (deduplicated) as retrieved context
```

### Complexity Analysis

| Phase | Standard RAG | HyDE | HyPE |
|-------|-------------|------|------|
| Index-time LLM calls | 0 | 0 | M (one per chunk) |
| Index-time embeddings | M | M | M × N |
| Query-time LLM calls | 0 | 1 | **0** |
| Query-time embeddings | 1 | 1 | 1 |
| FAISS search space | M | M | M × N |

HyPE **front-loads** all LLM computation to indexing time, making query-time as cheap as
standard RAG while benefiting from the semantic alignment of question-to-question matching.

---
# Part 5 — Implementation from Scratch

## Step 1: Hypothetical Question Generation

For each chunk, we prompt Qwen3-8B to generate 3-5 diverse questions.

In [ ]:
def generate_questions_for_chunk(chunk_text, n_questions=4):
    """Use the LLM to generate hypothetical questions for a text chunk."""
    prompt = (
        f"You are an expert at creating search queries. Given the following text passage, "
        f"generate exactly {n_questions} diverse questions that this passage could answer. "
        f"Each question should be on its own line. Do not number them. Do not add any "
        f"preamble or explanation — output ONLY the questions.\n\n"
        f"Passage:\n{chunk_text}"
    )
    messages = [{"role": "user", "content": prompt}]
    raw = generate(messages, max_new_tokens=256, temperature=0.7)

    # Parse: one question per non-empty line, strip numbering artifacts
    questions = []
    for line in raw.strip().split("\n"):
        line = line.strip()
        line = re.sub(r"^\d+[\.\)\-]\s*", "", line)  # strip "1. ", "2) " etc.
        line = re.sub(r"^[-•*]\s*", "", line)  # strip bullet markers
        if line and line.endswith("?"):
            questions.append(line)
    return questions[:n_questions]

# Demo: generate questions for the first chunk
demo_questions = generate_questions_for_chunk(all_chunks[0])
print(f"Chunk 0 [{chunk_metadata[0]}]:")
print(textwrap.fill(all_chunks[0], width=90))
print(f"\nGenerated {len(demo_questions)} questions:")
for q in demo_questions:
    print(f"  → {q}")

## Step 2: Generate Questions for All Chunks

We loop over every chunk, generating hypothetical questions. This is the **offline
indexing cost** — it only happens once.

In [ ]:
N_QUESTIONS = 4  # questions per chunk

print(f"Generating {N_QUESTIONS} questions per chunk for {len(all_chunks)} chunks …")
start_time = time.time()

hype_questions = []       # flat list of all generated questions
hype_question_to_chunk = []  # parallel list: index of the source chunk

for idx, chunk in enumerate(all_chunks):
    qs = generate_questions_for_chunk(chunk, n_questions=N_QUESTIONS)
    for q in qs:
        hype_questions.append(q)
        hype_question_to_chunk.append(idx)
    if (idx + 1) % 5 == 0 or idx == len(all_chunks) - 1:
        print(f"  Processed {idx + 1}/{len(all_chunks)} chunks  "
              f"({len(hype_questions)} questions so far)")

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s — {len(hype_questions)} total questions generated")
print(f"Average questions per chunk: {len(hype_questions)/len(all_chunks):.1f}")

## Step 3: Embed Questions & Build HyPE Index

In [ ]:
print("Embedding all hypothetical questions …")
question_embeddings = embed_model.encode(
    hype_questions, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)

# Build FAISS index
hype_index = faiss.IndexFlatIP(dim)
hype_index.add(question_embeddings)

print(f"HyPE FAISS index: {hype_index.ntotal} question vectors, dim={dim}")
print(f"Compared to standard index: {standard_index.ntotal} chunk vectors")
print(f"Expansion factor: {hype_index.ntotal / standard_index.ntotal:.1f}x")

## Steps 4-6: Query-Time Retrieval

At query time we:
1. Embed the user query (same model, same space as the questions).
2. Search the **HyPE question index** for nearest neighbors.
3. Map back to source chunks and deduplicate.

In [ ]:
def hype_retrieve(query, k=5, dedup=True):
    """
    HyPE retrieval: embed query, search question index, return source chunks.

    Parameters
    ----------
    query : str
    k : int – how many nearest questions to retrieve
    dedup : bool – deduplicate chunks (multiple questions may map to same chunk)

    Returns
    -------
    list of (chunk_text, doc_title, score, matched_question)
    """
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = hype_index.search(q_emb, k)

    results = []
    seen_chunks = set()
    for j, qi in enumerate(indices[0]):
        chunk_idx = hype_question_to_chunk[qi]
        if dedup and chunk_idx in seen_chunks:
            continue
        seen_chunks.add(chunk_idx)
        results.append((
            all_chunks[chunk_idx],
            chunk_metadata[chunk_idx],
            float(scores[0][j]),
            hype_questions[qi],
        ))
    return results

# Quick test
test_query = "How does the learning rate affect gradient descent?"
results = hype_retrieve(test_query, k=5)
print(f"Query: {test_query}\n")
for rank, (chunk, title, score, matched_q) in enumerate(results, 1):
    print(f"  Rank {rank}  [{title}]  score={score:.4f}")
    print(f"    Matched question: {matched_q}")
    print(f"    Chunk: {chunk[:100]}…\n")

---
# Part 6 — Question Quality Analysis

Good question generation is the backbone of HyPE. Let's analyze the quality of
the generated questions along three axes:

1. **Diversity** — Do questions cover different aspects of the chunk?
2. **Faithfulness** — Are questions actually answerable from the chunk?
3. **Coverage** — Across all chunks of a document, do the questions cover all key facts?

In [ ]:
# Show all generated questions grouped by chunk for one document
target_doc = "Transformer Architecture"

print(f"=== Questions for document: '{target_doc}' ===\n")
for idx, (chunk, title) in enumerate(zip(all_chunks, chunk_metadata)):
    if title != target_doc:
        continue
    # Gather questions for this chunk
    qs = [hype_questions[i] for i, ci in enumerate(hype_question_to_chunk) if ci == idx]
    print(f"Chunk {idx}:")
    print(textwrap.fill(chunk, width=90))
    print(f"  Questions ({len(qs)}):")
    for q in qs:
        print(f"    → {q}")
    print()

In [ ]:
# ── Diversity analysis: measure average pairwise cosine distance between questions per chunk ──
from collections import defaultdict

chunk_to_questions = defaultdict(list)
for i, ci in enumerate(hype_question_to_chunk):
    chunk_to_questions[ci].append(i)

diversities = []
for ci, q_indices in chunk_to_questions.items():
    if len(q_indices) < 2:
        continue
    q_embs = question_embeddings[q_indices]
    # Pairwise cosine similarity (already normalized)
    sim_matrix = q_embs @ q_embs.T
    n = len(q_indices)
    # Average off-diagonal similarity
    avg_sim = (sim_matrix.sum() - n) / (n * (n - 1))
    diversities.append(1.0 - avg_sim)  # diversity = 1 - similarity

avg_diversity = np.mean(diversities)
print(f"Question diversity analysis across {len(diversities)} chunks:\n")
print(f"  Mean intra-chunk diversity:  {avg_diversity:.4f}")
print(f"  Min diversity:               {min(diversities):.4f}")
print(f"  Max diversity:               {max(diversities):.4f}")
print(f"\n  (1.0 = completely different questions, 0.0 = identical questions)")
print(f"  Diversity > 0.2 is generally good for retrieval coverage.")

In [ ]:
# ── Coverage: for each document, can we retrieve ALL its chunks via questions? ──
print("Coverage test: querying with document-specific questions\n")

for doc in KNOWLEDGE_BASE:
    doc_chunk_indices = [i for i, t in enumerate(chunk_metadata) if t == doc["title"]]
    # Use one question from each chunk to query
    retrieved_chunk_indices = set()
    for ci in doc_chunk_indices:
        qs = [hype_questions[i] for i, c in enumerate(hype_question_to_chunk) if c == ci]
        if qs:
            results = hype_retrieve(qs[0], k=3, dedup=True)
            for r_chunk, r_title, _, _ in results:
                if r_title == doc["title"]:
                    r_idx = all_chunks.index(r_chunk)
                    retrieved_chunk_indices.add(r_idx)
    coverage = len(retrieved_chunk_indices) / len(doc_chunk_indices) * 100
    print(f"  {doc['title']:40s}  chunks={len(doc_chunk_indices)}  "
          f"covered={len(retrieved_chunk_indices)}  coverage={coverage:.0f}%")

---
# Part 7 — Comparison: Standard vs HyDE vs HyPE

We test all three retrieval strategies on the **same set of queries** and compare which
chunks they retrieve, their relevance scores, and retrieval latency.

In [ ]:
TEST_QUERIES = [
    "What optimizer combines momentum with adaptive learning rates?",
    "How does self-attention work in transformers?",
    "What is the difference between precision and recall?",
    "How does dropout regularize a neural network?",
    "What are the stages of a RAG pipeline?",
    "Why are residual connections important in CNNs?",
]

print(f"Prepared {len(TEST_QUERIES)} test queries for three-way comparison.")

In [ ]:
# ── Three-way comparison ──
import time

print(f"{'Query':<55} {'Method':<10} {'Top-1 Doc':<35} {'Score':>7} {'Time':>8}")
print("─" * 120)

for query in TEST_QUERIES:
    # Standard
    t0 = time.time()
    std_results = standard_retrieve(query, k=1)
    std_time = time.time() - t0
    print(f"{query[:53]:<55} {'Standard':<10} {std_results[0][1]:<35} "
          f"{std_results[0][2]:>7.4f} {std_time*1000:>6.1f}ms")

    # HyDE
    t0 = time.time()
    hyde_results, hypo_doc = hyde_retrieve(query, k=1)
    hyde_time = time.time() - t0
    print(f"{'':<55} {'HyDE':<10} {hyde_results[0][1]:<35} "
          f"{hyde_results[0][2]:>7.4f} {hyde_time*1000:>6.1f}ms")

    # HyPE
    t0 = time.time()
    hype_results = hype_retrieve(query, k=1)
    hype_time = time.time() - t0
    print(f"{'':<55} {'HyPE':<10} {hype_results[0][1]:<35} "
          f"{hype_results[0][2]:>7.4f} {hype_time*1000:>6.1f}ms")

    print("─" * 120)

In [ ]:
# ── Detailed view of one query ──
query = "What optimizer combines momentum with adaptive learning rates?"
print(f"Query: {query}\n")

print("=== Standard RAG (top-3) ===")
for chunk, title, score in standard_retrieve(query, k=3):
    print(f"  [{title}] score={score:.4f}")
    print(f"  {textwrap.shorten(chunk, width=100)}\n")

print("=== HyDE (top-3) ===")
hyde_res, hypo = hyde_retrieve(query, k=3)
print(f"  Hypothetical document: {textwrap.shorten(hypo, width=100)}\n")
for chunk, title, score in hyde_res:
    print(f"  [{title}] score={score:.4f}")
    print(f"  {textwrap.shorten(chunk, width=100)}\n")

print("=== HyPE (top-3) ===")
for chunk, title, score, matched_q in hype_retrieve(query, k=5)[:3]:
    print(f"  [{title}] score={score:.4f}")
    print(f"  Matched question: {matched_q}")
    print(f"  {textwrap.shorten(chunk, width=100)}\n")

### Latency Analysis

Let's measure average query latency for each method over all test queries.

In [ ]:
latencies = {"Standard": [], "HyDE": [], "HyPE": []}

for query in TEST_QUERIES:
    t0 = time.time(); standard_retrieve(query, k=3); latencies["Standard"].append(time.time() - t0)
    t0 = time.time(); hyde_retrieve(query, k=3); latencies["HyDE"].append(time.time() - t0)
    t0 = time.time(); hype_retrieve(query, k=5); latencies["HyPE"].append(time.time() - t0)

print("Average query latency:\n")
for method, times in latencies.items():
    avg_ms = np.mean(times) * 1000
    print(f"  {method:<10}  {avg_ms:>8.1f} ms")

print(f"\nHyDE/Standard ratio:  {np.mean(latencies['HyDE'])/np.mean(latencies['Standard']):.1f}x slower")
print(f"HyPE/Standard ratio:  {np.mean(latencies['HyPE'])/np.mean(latencies['Standard']):.1f}x")
print(f"\nHyPE achieves question-to-question matching with latency comparable to standard RAG.")

### Relevance Scoring

We use the LLM as a judge to score the relevance of top-1 retrieved chunks on a
1-5 scale for each method.

In [ ]:
def judge_relevance(query, chunk_text):
    """Use the LLM to score relevance of a chunk to a query on a 1-5 scale."""
    prompt = (
        f"Rate how relevant the following passage is for answering the question. "
        f"Respond with ONLY a single integer from 1 (irrelevant) to 5 (perfectly relevant).\n\n"
        f"Question: {query}\n\nPassage: {chunk_text}"
    )
    messages = [{"role": "user", "content": prompt}]
    raw = generate(messages, max_new_tokens=10, temperature=0.1)
    # Extract first digit
    match = re.search(r"[1-5]", raw)
    return int(match.group()) if match else 3

scores = {"Standard": [], "HyDE": [], "HyPE": []}

print("Scoring relevance with LLM-as-judge …\n")
for query in TEST_QUERIES:
    std = standard_retrieve(query, k=1)
    hyde_res, _ = hyde_retrieve(query, k=1)
    hype_res = hype_retrieve(query, k=1)

    s_std = judge_relevance(query, std[0][0])
    s_hyde = judge_relevance(query, hyde_res[0][0])
    s_hype = judge_relevance(query, hype_res[0][0])

    scores["Standard"].append(s_std)
    scores["HyDE"].append(s_hyde)
    scores["HyPE"].append(s_hype)

    print(f"  Q: {query[:60]}")
    print(f"    Standard={s_std}  HyDE={s_hyde}  HyPE={s_hype}")

print(f"\n{'Method':<10} {'Avg Relevance':>15}")
print("─" * 28)
for method, vals in scores.items():
    print(f"  {method:<10} {np.mean(vals):>12.2f}")

---
# Part 8 — Hybrid Approach: Document + Question Index

## Motivation

Standard RAG and HyPE each have strengths:
- **Standard** excels when the query uses the same vocabulary as the document
  (exact keyword matches, technical terms).
- **HyPE** excels when the query is phrased differently from the document
  (paraphrasing, high-level questions).

A **hybrid** approach searches **both** indices and fuses the results via
**Reciprocal Rank Fusion (RRF)**, a simple but effective score-combination
method that doesn't require score normalization.

### Reciprocal Rank Fusion

For each document that appears in any ranking, RRF computes:

$$\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}$$

where *k* is a constant (typically 60) that dampens the influence of high-rank items.

In [ ]:
def rrf_score(rank, k=60):
    """Reciprocal Rank Fusion score for a given rank (0-indexed)."""
    return 1.0 / (k + rank + 1)

def hybrid_retrieve(query, k_per_index=5, final_k=3):
    """
    Hybrid retrieval: search both standard chunk index and HyPE question index,
    then fuse results with Reciprocal Rank Fusion (RRF).
    """
    # ── Search standard index ──
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, std_indices = standard_index.search(q_emb, k_per_index)

    # ── Search HyPE index ──
    _, hype_indices = hype_index.search(q_emb, k_per_index * 2)  # more because we dedup

    # Map HyPE question indices → chunk indices (dedup)
    hype_chunk_ranks = {}
    for rank, qi in enumerate(hype_indices[0]):
        ci = hype_question_to_chunk[qi]
        if ci not in hype_chunk_ranks:
            hype_chunk_ranks[ci] = rank

    # ── RRF fusion ──
    chunk_scores = {}
    # Standard contributions
    for rank, ci in enumerate(std_indices[0]):
        chunk_scores[ci] = chunk_scores.get(ci, 0) + rrf_score(rank)
    # HyPE contributions
    for ci, rank in hype_chunk_ranks.items():
        chunk_scores[ci] = chunk_scores.get(ci, 0) + rrf_score(rank)

    # Sort by fused score
    ranked = sorted(chunk_scores.items(), key=lambda x: x[1], reverse=True)[:final_k]

    results = []
    for ci, score in ranked:
        results.append((all_chunks[ci], chunk_metadata[ci], score))
    return results

# Test
query = "What optimizer combines momentum with adaptive learning rates?"
print(f"Query: {query}\n")
print("=== Hybrid (RRF) top-3 ===")
for chunk, title, score in hybrid_retrieve(query):
    print(f"  [{title}] rrf_score={score:.4f}")
    print(f"  {textwrap.shorten(chunk, width=100)}\n")

In [ ]:
# ── Compare all four methods ──
print(f"{'Query':<55} {'Std':^8} {'HyDE':^8} {'HyPE':^8} {'Hybrid':^8}")
print("─" * 95)

for query in TEST_QUERIES:
    std = standard_retrieve(query, k=1)[0][1]
    hyde_r, _ = hyde_retrieve(query, k=1)
    hyde_doc = hyde_r[0][1]
    hype_doc = hype_retrieve(query, k=1)[0][1]
    hyb_doc = hybrid_retrieve(query, final_k=1)[0][1]

    # Truncate titles for display
    def trunc(s, n=8): return s[:n]
    print(f"{query[:53]:<55} {trunc(std):^8} {trunc(hyde_doc):^8} "
          f"{trunc(hype_doc):^8} {trunc(hyb_doc):^8}")

In [ ]:
# ── Add hybrid to relevance scoring ──
scores["Hybrid"] = []

print("Scoring hybrid relevance …\n")
for query in TEST_QUERIES:
    hyb = hybrid_retrieve(query, final_k=1)
    s_hyb = judge_relevance(query, hyb[0][0])
    scores["Hybrid"].append(s_hyb)
    print(f"  Q: {query[:60]}  Hybrid={s_hyb}")

print(f"\n{'Method':<10} {'Avg Relevance':>15}")
print("─" * 28)
for method, vals in scores.items():
    print(f"  {method:<10} {np.mean(vals):>12.2f}")

---
# Part 9 — Synthesis: When to Use What

## HyPE Front-loads Computation

The fundamental insight: **HyPE moves the LLM cost from query time to index time.**

```
Standard RAG:  Index cost = O(M·embed)    Query cost = O(embed + search)
HyDE:          Index cost = O(M·embed)    Query cost = O(LLM + embed + search)
HyPE:          Index cost = O(M·LLM + M·N·embed)   Query cost = O(embed + search)
```

This makes HyPE ideal for **read-heavy** workloads — systems where the corpus changes
infrequently but queries come at high volume.

## Decision Framework

| Scenario | Best approach | Why |
|----------|--------------|-----|
| Static corpus, high query volume | **HyPE** | Pay LLM cost once at indexing; queries are fast |
| Dynamic corpus, frequent updates | **Standard** or **HyDE** | Re-indexing with HyPE is expensive |
| Complex multi-hop queries | **HyDE + re-ranker** | Query-time LLM can adapt to each complex query |
| Latency-critical applications | **HyPE** or **Standard** | No query-time LLM dependency |
| Best overall quality | **Hybrid (Standard + HyPE)** | Fuse vocabulary-match and semantic-match signals |

## Tradeoffs

### HyPE advantages
- ✅ Zero query-time LLM calls → lower latency, lower cost
- ✅ Question-to-question matching → better semantic alignment
- ✅ Multiple questions per chunk → multi-faceted coverage
- ✅ Index once, serve forever (for static corpora)

### HyPE limitations
- ⚠️ Indexing is expensive (LLM call per chunk)
- ⚠️ Question quality depends on the LLM
- ⚠️ Larger FAISS index (N× more vectors)
- ⚠️ Re-indexing needed when corpus changes
- ⚠️ Cannot adapt to novel query patterns not anticipated during indexing

In [ ]:
# ── Index size comparison ──
standard_size = standard_index.ntotal * dim * 4  # float32
hype_size = hype_index.ntotal * dim * 4

print("Index Size Comparison:\n")
print(f"  Standard index:  {standard_index.ntotal:>6} vectors  →  {standard_size/1024:.1f} KB")
print(f"  HyPE index:      {hype_index.ntotal:>6} vectors  →  {hype_size/1024:.1f} KB")
print(f"  Expansion:       {hype_index.ntotal / standard_index.ntotal:.1f}x vectors, "
      f"{hype_size / standard_size:.1f}x storage")
print(f"\nFor production with millions of chunks, consider using FAISS IVF or HNSW "
      f"indices to keep search fast despite the larger index size.")

## End-to-End: HyPE-Powered RAG

Let's put it all together — retrieve with HyPE, then generate an answer with Qwen3.

In [ ]:
def hype_rag(query, k=5, final_chunks=3):
    """Full HyPE-powered RAG: retrieve with HyPE, generate answer with LLM."""
    results = hype_retrieve(query, k=k)[:final_chunks]

    context = "\n\n---\n\n".join([chunk for chunk, _, _, _ in results])

    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant. Answer the question using ONLY the provided context. "
            "If the context doesn't contain enough information, say so."
        )},
        {"role": "user", "content": (
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            f"Answer:"
        )},
    ]
    answer = generate(messages, max_new_tokens=300, temperature=0.3)
    return answer, results

# Test
query = "How does Adam optimizer work and why is it better than basic SGD?"
answer, sources = hype_rag(query)

print(f"Query: {query}\n")
print(f"Answer:\n{textwrap.fill(answer, width=90)}\n")
print("Sources:")
for chunk, title, score, matched_q in sources:
    print(f"  [{title}] via question: {matched_q}")

In [ ]:
# Another end-to-end example
query = "What metrics should I use to evaluate a classifier on an imbalanced dataset?"
answer, sources = hype_rag(query)

print(f"Query: {query}\n")
print(f"Answer:\n{textwrap.fill(answer, width=90)}\n")
print("Sources:")
for chunk, title, score, matched_q in sources:
    print(f"  [{title}] score={score:.4f}")
    print(f"    Question: {matched_q}")

---
# Summary

## What We Built

| Component | Implementation |
|-----------|---------------|
| LLM | Qwen3-8B (4-bit NF4) — local, no API |
| Embeddings | BAAI/bge-base-en-v1.5 via sentence-transformers |
| Vector Store | FAISS (IndexFlatIP with normalized vectors = cosine) |
| Question Generation | Prompted Qwen3 to generate 4 questions per chunk |
| Retrieval | Query → embed → search question index → map to chunks |
| Comparison | Standard vs HyDE vs HyPE vs Hybrid (RRF) |

## Key Takeaways

1. **HyPE is the inverse of HyDE** — generate questions at index time, not documents at
   query time.

2. **Question-to-question matching** works better than question-to-document because both
   texts share the same semantic register.

3. **HyPE has zero query-time LLM cost** — all generation is front-loaded to indexing.

4. **The hybrid approach** (Standard + HyPE with RRF) often gives the best results by
   combining exact vocabulary match with semantic question matching.

5. **Tradeoff axis**: HyPE trades *indexing cost* for *query-time speed and quality*.
   Ideal for static, read-heavy workloads.

## Further Reading

- [HyPE preprint (SSRN 5139335)](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5139335)
- [HyDE paper: Gao et al., 2023](https://arxiv.org/abs/2212.10496)
- [BGE embeddings](https://huggingface.co/BAAI/bge-base-en-v1.5)
- [FAISS documentation](https://github.com/facebookresearch/faiss)